# 01b Inspect the splits and the attacks
Label distribution per client, the label-flipped client, and the triggers (the raw material for the planned interactive sliders).

In [ ]:
# find the repo root (works from notebooks/<group>/, from the repo root, on Kaggle and on Colab) and set up the folders
import os, sys, shutil
from pathlib import Path
KAGGLE_REPO = Path("/kaggle/input/datasets/sakhiurrahman/fedupds/FedUP_Reproduction")   # your Kaggle dataset (edit if you move it)
def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working/FedUP_Reproduction"), Path("/content/FedUP_Reproduction"), KAGGLE_REPO]:
        if (p / "fedup_repro").exists():
            return p
    kin = Path("/kaggle/input")
    if kin.exists():                                    # any other layout of the input dataset, up to 6 levels deep
        for dp, dn, _ in os.walk(kin):
            if "fedup_repro" in dn:
                return Path(dp)
            if len(Path(dp).relative_to(kin).parts) > 6:
                dn[:] = []
    return None
ROOT = _find_repo()
assert ROOT is not None, "FedUP_Reproduction folder not found: upload it (Kaggle: as a Dataset) or cd into it"
if str(ROOT).startswith("/kaggle/input"):              # input datasets are read-only: work on a copy of the code ...
    SRC, ROOT = ROOT, Path("/kaggle/working") / ROOT.name
    if not ROOT.exists():
        shutil.copytree(SRC, ROOT, ignore=lambda d, names: [n for n in names if n == "raw" and Path(d).name == "data"])
    os.environ["FEDUP_BUNDLED_RAW"] = str(SRC / "data" / "raw")      # ... and LINK (not copy) the datasets that ship with it
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from fedup_repro import colab
PERSIST_ROOT = None        # Colab + Drive: "/content/drive/MyDrive/fedup_store" keeps data/ and artifacts/ across sessions (mount Drive first)
print("platform:", colab.setup_dirs(PERSIST_ROOT), "| repo:", ROOT)

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from fedup_repro.config import make_config
from fedup_repro import data as D
BENCH = "CIFAR10-ResNet18"                     # change me
cfg = make_config(BENCH, "sample")
fed = D.build_fed_data(cfg, 0); view = D.make_view(cfg, fed, 0)
K = cfg["num_classes"]
fig, ax = plt.subplots(figsize=(8, 3.5)); bottom = np.zeros(len(view["clients"]))
for k in range(K):
    h = np.array([(c["y"] == k).sum().item() for c in view["clients"]]); ax.bar(range(len(h)), h, bottom=bottom); bottom += h
ax.set_xlabel("client"); ax.set_ylabel("samples"); ax.set_title(f"{BENCH}: label distribution (Dirichlet {cfg['alpha']})"); plt.show()

In [ ]:
# clean vs triggered sample (images only): sample trigger (single pixel) and FuGuard-style patch
import torch
c0 = view["clients"][0]; x = c0["x"][:1]
dev = torch.device("cpu")
clean = D.prep(x, cfg, dev)
trig_pixel = D.make_trigger(cfg, "sample")(clean)
cfg_bd = make_config(BENCH, "client_backdoor"); trig_patch = D.make_trigger(cfg_bd, "client_backdoor")(clean)
def show(t, ax, title):
    mean, std = D.norm_stats(cfg)
    im = (t[0] * torch.tensor(std).view(-1,1,1) + torch.tensor(mean).view(-1,1,1)).clamp(0, 1)
    ax.imshow(im.permute(1, 2, 0).squeeze().numpy(), cmap="gray"); ax.set_title(title); ax.axis("off")
fig, ax = plt.subplots(1, 3, figsize=(7, 2.6)); show(clean, ax[0], "clean"); show(trig_pixel, ax[1], "sample trigger"); show(trig_patch, ax[2], "client_backdoor patch"); plt.show()